# Pilot 2 on the pod

Run cell 1, send Claude the lock file, wait for the go-ahead, run cell 2. Cell 3 only if cell 2 asks for it. Each cell beeps when it ends: three rising beeps = done, two low beeps = stopped (scroll up for the error). Every step resumes where it left off, so after a crash just run the same cell again.


In [ ]:
# ===== CELL 1: setup, quick check, tune phase, format choice (about 45-60 minutes) =====
# When you hear three rising beeps: download /workspace/pilot2_lock.tar.gz and send it to
# Claude. Claude commits it to GitHub (the lock). Leave the pod running, then run cell 2.
import os, subprocess, json, numpy as np
from IPython.display import Audio, display

REPO = "/workspace/llm-value-trade-offs"
P2 = f"{REPO}/pilot2-value-gate"
os.environ["HF_HOME"] = "/workspace/hf"          # models are cached on the volume disk

def beep(ok=True):
    """Three rising beeps = done. Two low beeps = something failed."""
    sr = 22050
    t = np.linspace(0, 0.22, int(sr * 0.22), False)
    gap = np.zeros(int(sr * 0.08))
    freqs = [523, 659, 784] if ok else [220, 165]
    display(Audio(np.concatenate([np.concatenate([0.5 * np.sin(2 * np.pi * f * t), gap]) for f in freqs]),
                  rate=sr, autoplay=True))

def run(cmd, log_file=None):
    """Run a shell command and show its output live (or send it to a file, unseen)."""
    print(f"\n>>> {cmd}", flush=True)
    if log_file:
        cmd = f"({cmd}) > {log_file} 2>&1"
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Failed: {cmd}" + (f" (see {log_file})" if log_file else ""))

def run_all(steps):
    try:
        for s in steps:
            run(*s) if isinstance(s, tuple) else run(s)
    except Exception as e:
        print(f"\n*** STOPPED: {e}")
        beep(False)
        return False
    beep(True)
    return True

run_all([
    f"cd /workspace && (test -d {REPO} || git clone https://github.com/JasperBlank/llm-value-trade-offs.git)",
    f"cd {REPO} && git pull --ff-only && git log -1 --format='Running commit %h: %s'",
    f"cd {P2} && pip install -q -r requirements.txt",
    "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader && df -h /workspace | tail -1",
    f"cd {P2} && python run.py --phase tune --model quick --quick",     # small model: is everything working?
    f"cd {P2} && python run.py --phase tune --model 8b",
    f"cd {P2} && python run.py --phase tune --model 14b",
    f"cd {P2} && python select_format.py",                              # applies the pre-registered rule
    f"cd {P2}/results/real && tar czf /workspace/pilot2_lock.tar.gz locked_format.json tune_report.md && ls -la /workspace/pilot2_lock.tar.gz",
])


In [ ]:
# ===== CELL 2: test phase and goal module on 4B, 8B, 14B (about 30-45 minutes) =====
# Run only after Claude has said the lock is on GitHub. Results are not shown here.
# When you hear three rising beeps: download /workspace/pilot2_results.tar.gz, send it to
# Claude, and read the last line: it says whether a 32B run is needed.
import os, subprocess, json, numpy as np
from IPython.display import Audio, display

REPO = "/workspace/llm-value-trade-offs"
P2 = f"{REPO}/pilot2-value-gate"
os.environ["HF_HOME"] = "/workspace/hf"          # models are cached on the volume disk

def beep(ok=True):
    """Three rising beeps = done. Two low beeps = something failed."""
    sr = 22050
    t = np.linspace(0, 0.22, int(sr * 0.22), False)
    gap = np.zeros(int(sr * 0.08))
    freqs = [523, 659, 784] if ok else [220, 165]
    display(Audio(np.concatenate([np.concatenate([0.5 * np.sin(2 * np.pi * f * t), gap]) for f in freqs]),
                  rate=sr, autoplay=True))

def run(cmd, log_file=None):
    """Run a shell command and show its output live (or send it to a file, unseen)."""
    print(f"\n>>> {cmd}", flush=True)
    if log_file:
        cmd = f"({cmd}) > {log_file} 2>&1"
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Failed: {cmd}" + (f" (see {log_file})" if log_file else ""))

def run_all(steps):
    try:
        for s in steps:
            run(*s) if isinstance(s, tuple) else run(s)
    except Exception as e:
        print(f"\n*** STOPPED: {e}")
        beep(False)
        return False
    beep(True)
    return True

ok = run_all([
    # The locked format on GitHub must be identical to the one this pod chose.
    f"cd {REPO} && git fetch -q origin main && git show origin/main:pilot2-value-gate/results/real/locked_format.json > /tmp/lock_on_github.json "
    f"&& cmp /tmp/lock_on_github.json {P2}/results/real/locked_format.json && echo 'Lock found on GitHub and identical.'",
    f"cd {P2} && python run.py --phase test --model 4b",
    f"cd {P2} && python run.py --phase goal --model 4b",
    f"cd {P2} && python run.py --phase test --model 8b",
    f"cd {P2} && python run.py --phase goal --model 8b",
    f"cd {P2} && python run.py --phase test --model 14b",
    f"cd {P2} && python run.py --phase goal --model 14b",
    (f"cd {P2} && python analyze.py", "/workspace/analyze.log"),       # output goes to a file, unseen
    f"cd {P2}/results && tar czf /workspace/pilot2_results.tar.gz real && ls -la /workspace/pilot2_results.tar.gz",
])
if ok:
    needed = json.load(open(f"{P2}/results/real/analysis_14b.json"))["borderline"]
    print("\n32B run needed:", "YES (see cell 3)" if needed else "no. You can terminate the pod after downloading.")


In [ ]:
# ===== CELL 3: ONLY if cell 2 said "32B run needed: YES" =====
# Run this on a NEW pod with an A100 80 GB and a volume disk of at least 100 GB (not on the
# A6000). The locked format comes from GitHub. About 1 hour.
# When you hear three rising beeps: download /workspace/pilot2_results_32b.tar.gz and send it to Claude.
import os, subprocess, json, numpy as np
from IPython.display import Audio, display

REPO = "/workspace/llm-value-trade-offs"
P2 = f"{REPO}/pilot2-value-gate"
os.environ["HF_HOME"] = "/workspace/hf"          # models are cached on the volume disk

def beep(ok=True):
    """Three rising beeps = done. Two low beeps = something failed."""
    sr = 22050
    t = np.linspace(0, 0.22, int(sr * 0.22), False)
    gap = np.zeros(int(sr * 0.08))
    freqs = [523, 659, 784] if ok else [220, 165]
    display(Audio(np.concatenate([np.concatenate([0.5 * np.sin(2 * np.pi * f * t), gap]) for f in freqs]),
                  rate=sr, autoplay=True))

def run(cmd, log_file=None):
    """Run a shell command and show its output live (or send it to a file, unseen)."""
    print(f"\n>>> {cmd}", flush=True)
    if log_file:
        cmd = f"({cmd}) > {log_file} 2>&1"
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Failed: {cmd}" + (f" (see {log_file})" if log_file else ""))

def run_all(steps):
    try:
        for s in steps:
            run(*s) if isinstance(s, tuple) else run(s)
    except Exception as e:
        print(f"\n*** STOPPED: {e}")
        beep(False)
        return False
    beep(True)
    return True

run_all([
    f"cd /workspace && (test -d {REPO} || git clone https://github.com/JasperBlank/llm-value-trade-offs.git)",
    f"cd {REPO} && git pull --ff-only && git log -1 --format='Running commit %h: %s'",
    f"cd {P2} && pip install -q -r requirements.txt",
    f"test -f {P2}/results/real/locked_format.json && echo 'Locked format found.'",
    "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader && df -h /workspace | tail -1",
    f"cd {P2} && python run.py --phase test --model 32b",
    f"cd {P2} && python run.py --phase goal --model 32b",
    (f"cd {P2} && python analyze.py", "/workspace/analyze.log"),
    f"cd {P2}/results && tar czf /workspace/pilot2_results_32b.tar.gz real/test_32b.jsonl real/goal_32b.jsonl real/analysis_32b.json real/report_32b.md && ls -la /workspace/pilot2_results_32b.tar.gz",
])
